# Notebook 3: Landsat 8 Feature Extraction

## Purpose
Extracts 6 spectral bands and calculates 12 water quality indices from Landsat 8 satellite imagery for all 9,319 water quality samples via Microsoft Planetary Computer.

## Input
- `unique_samples_for_extraction.csv` (9,319 unique location-date combinations)

## Process
- Queries Landsat 8 scenes with <10% cloud cover
- Selects single closest scene to sample date
- Extracts median reflectance across pixels within 100m buffer around sample point
- Calculates 12 indices (NDVI, EVI, SAVI, MNDWI, chlorophyll proxy, turbidity, etc.)

## Output
- `landsat_features.csv` (9,319 rows × 21 columns: Lat, Lon, Date, 6 bands, 12 indices, 3 identifiers)
- ~12% of samples have missing data (primarily 2011-2012 pre-Landsat 8 launch, or persistent cloud cover)

## Runtime
⏱️ **~11 hours** (API rate limits, cloud-free scene availability)

In [5]:
# Install required packages with NumPy 1.x for compatibility
!pip install "numpy<2.0" pandas xarray scipy pystac-client planetary-computer tqdm \
             dask distributed scikit-learn lightgbm xgboost folium plotly joblib dill adlfs

  Using cached adlfs-2026.4.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached azure_core-1.39.0-py3-none-any.whl.metadata (48 kB)
  Using cached azure_datalake_store-0.0.53-py2.py3-none-any.whl.metadata (19 kB)
  Using cached azure_identity-1.25.3-py3-none-any.whl.metadata (91 kB)
  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
  Using cached msal-1.36.0-py3-none-any.whl.metadata (11 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached msal_extensions-1.3.1-py3-none-any.whl.metadata (7.8 kB)
Using cached adlfs-2026.4.0-py3-none-any.whl (46 kB)
Using cached azure_core-1.39.0-py3-none-any.whl (218 kB)
Using cached azure_datalake_store-0.0.53-py2.py3-none-any.whl (55 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)
Using cached azure_identity-1.25.3-py3-none-any.whl (192 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)
Using cached msal-1.36.0-py3-none-any.whl (121 kB)
Using cached msal_extensions-1.3.1

Cell 2: Install Libraries

In [17]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os

# Planetary Computer and STAC
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

# Progress tracking
from tqdm import tqdm
tqdm.pandas()

# Download utilities
import base64
from IPython.display import HTML, display

print("All libraries imported successfully")
print(f"Working directory: {os.getcwd()}")

All libraries imported successfully
Working directory: /Users/juliusrashba/AI DSS PROJECT


Cell 3: Load input data

In [19]:
# Load unique samples
input_df = pd.read_csv('unique_samples_for_extraction.csv')

print(f"Total samples: {len(input_df):,}")
print(f"Unique locations: {input_df[['Latitude', 'Longitude']].drop_duplicates().shape[0]:,}")
print(f"Date range: {input_df['Sample Date'].min()} to {input_df['Sample Date'].max()}")
print(f"\nFirst few rows:")
display(input_df.head())
print(f"\nColumns: {list(input_df.columns)}")
print(f"\nMissing values:\n{input_df.isnull().sum()}")

Total samples: 9,319
Unique locations: 162
Date range: 2011-01-02 to 2015-12-31

First few rows:


,Latitude,Longitude,Sample Date
0,-34.405833,19.600556,2011-07-06
1,-34.405833,19.600556,2011-08-02
2,-34.405833,19.600556,2011-08-30
3,-34.405833,19.600556,2011-09-27
4,-34.405833,19.600556,2011-10-27



Columns: ['Latitude', 'Longitude', 'Sample Date']

Missing values:
Latitude       0
Longitude      0
Sample Date    0
dtype: int64


Cell 4: Extraction Function

In [21]:
# Initialize catalog once (shared across all calls)
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

def extract_landsat_bands(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')
    
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    
    # ±45 day window (90 days total)
    start_date = (date - timedelta(days=45)).strftime('%Y-%m-%d')
    end_date = (date + timedelta(days=45)).strftime('%Y-%m-%d')
    
    try:
        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime=f"{start_date}/{end_date}",  # ← DYNAMIC WINDOW
            query={"eo:cloud_cover": {"lt": 10}},
        )
        
        items = search.item_collection()
        
        if not items:
            # No scenes found
            return pd.Series({
                "nir": np.nan, "green": np.nan, "red": np.nan,
                "blue": np.nan, "swir16": np.nan, "swir22": np.nan
            })
        
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        
        # Select scene closest to sample date
        items_sorted = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items_sorted[0])
        
        # Load all 6 bands
        bands_of_interest = ["nir08", "green", "red", "blue", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        
        # Extract median values
        nir = data["nir08"].astype("float")
        green = data["green"].astype("float")
        red = data["red"].astype("float")
        blue = data["blue"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # Compute medians (skipna=True handles internal NaNs)
        median_nir = float(nir.median(skipna=True).values)
        median_green = float(green.median(skipna=True).values)
        median_red = float(red.median(skipna=True).values)
        median_blue = float(blue.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)
        
        # Replace 0 with NaN (invalid Landsat values)
        median_nir = median_nir if median_nir != 0 else np.nan
        median_green = median_green if median_green != 0 else np.nan
        median_red = median_red if median_red != 0 else np.nan
        median_blue = median_blue if median_blue != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "red": median_red,
            "blue": median_blue,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
        
    except Exception as e:
        # Handle any errors gracefully
        return pd.Series({
            "nir": np.nan, "green": np.nan, "red": np.nan,
            "blue": np.nan, "swir16": np.nan, "swir22": np.nan
        })

print("Landsat extraction function defined")
print("Connected to Microsoft Planetary Computer")
print("Data source: Landsat Collection 2 Level 2")

Landsat extraction function defined
Connected to Microsoft Planetary Computer
Data source: Landsat Collection 2 Level 2


Cell 5: Testing extraction b/ it takes a while make sure code is good

In [24]:
# Test on first 5 rows
print("Testing extraction on first 5 samples\n")

test_df = input_df.head(5).copy()
test_results = test_df.progress_apply(extract_landsat_bands, axis=1)

print("\nTest extraction complete\n")
print("Test Results:")
display(test_results)

# Check for any data
nan_count = test_results.isna().sum().sum()
total_values = test_results.size
valid_values = total_values - nan_count

print(f"\nData Quality:")
print(f"   Valid values: {valid_values}/{total_values} ({100*valid_values/total_values:.1f}%)")
print(f"   Missing values: {nan_count}/{total_values} ({100*nan_count/total_values:.1f}%)")

if valid_values > 0:
    print("\nExtraction function working correctly! Ready to process full dataset.")
else:
    print("\nWARNING: No valid data extracted. Check API connection and coordinates.")

Testing extraction on first 5 samples



100%|█████████████████████████████████████████████| 5/5 [00:04<00:00,  1.20it/s]


Test extraction complete

Test Results:


,nir,green,red,blue,swir16,swir22
0,NaN,NaN,NaN,NaN,NaN,NaN
1,15064.5,9899.5,10479.5,9162.0,17232.5,14168.5
2,6635.5,4251.0,4208.0,4019.5,6034.5,4820.5
3,18317.0,9145.5,8971.5,8456.0,13913.5,10727.5
4,18317.0,9145.5,8971.5,8456.0,13913.5,10727.5



Data Quality:
   Valid values: 24/30 (80.0%)
   Missing values: 6/30 (20.0%)

Extraction function working correctly! Ready to process full dataset.


Cell 6: Full extraction (in batches)

In [ ]:
import os
from tqdm import tqdm

# Configuration
BATCH_SIZE = 200
BATCH_DIR = "landsat_batches"
total_samples = len(input_df)
total_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

# Create batch directory
os.makedirs(BATCH_DIR, exist_ok=True)

print(f"Starting Landsat extraction: {total_samples:,} samples in {total_batches} batches\n")

# Check existing batches
existing_batches = set()
if os.path.exists(BATCH_DIR):
    for f in os.listdir(BATCH_DIR):
        if f.startswith('landsat_batch_') and f.endswith('.csv'):
            batch_start = f.split('_')[2]
            existing_batches.add(batch_start)
    if existing_batches:
        print(f"Found {len(existing_batches)} existing batches - will skip these\n")

# Process batches
for batch_num in range(total_batches):
    batch_start = batch_num * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_samples)
    batch_path = f"{BATCH_DIR}/landsat_batch_{batch_start}_{batch_end}.csv"
    
    # Skip if exists
    if str(batch_start) in existing_batches:
        print(f"Batch {batch_num+1}/{total_batches}: Skipped (already complete)")
        continue
    
    print(f"Batch {batch_num+1}/{total_batches} [{batch_start}-{batch_end}]:")
    
    try:
        batch_df = input_df.iloc[batch_start:batch_end].copy()
        
        # Extract with progress bar
        results = []
        for _, row in tqdm(batch_df.iterrows(), total=len(batch_df), leave=False):
            results.append(extract_landsat_bands(row))
        
        batch_results = pd.DataFrame(results)
        batch_results.to_csv(batch_path, index=False)
        
        # Stats
        valid_pct = 100 * (1 - batch_results.isna().sum().sum() / batch_results.size)
        print(f"  ✓ Saved - {valid_pct:.1f}% valid data\n")
        
    except Exception as e:
        print(f"  ✗ ERROR: {str(e)}\n  Re-run to resume\n")
        break

print(f"Extraction complete! Batches saved to '{BATCH_DIR}/' folder")

Starting Landsat extraction: 9,319 samples in 47 batches

Batch 1/47 [0-200]:


  ✓ Saved - 69.5% valid data

Batch 2/47 [200-400]:


  ✓ Saved - 72.0% valid data

Batch 3/47 [400-600]:


  ✓ Saved - 86.0% valid data

Batch 4/47 [600-800]:


  ✓ Saved - 77.0% valid data

Batch 5/47 [800-1000]:


...

 36%|██████████████▉                           | 71/200 [01:37<00:51,  2.49it/s]Aborting load due to failure while reading: https://landsateuwest.blob.core.windows.net/landsat-c2/level-2/standard/etm/2011/170/081/LE07_L2SP_170081_20110422_20200910_02_T1/LE07_L2SP_170081_20110422_20200910_02_T1_SR_B2.TIF?st=2026-04-29T22%3A17%3A34Z&se=2026-04-30T23%3A02%3A34Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-04-30T22%3A05%3A14Z&ske=2026-05-07T22%3A05%3A14Z&sks=b&skv=2025-07-05&sig=M4mGTdgR/vhHWkQmwczdi94qcm4ZskkNsTjasCgIMQs%3D:1


  ✓ Saved - 7.0% valid data

Batch 11/47 [2000-2200]:


  ✓ Saved - 82.0% valid data

Batch 12/47 [2200-2400]:


  ✓ Saved - 77.5% valid data

Batch 13/47 [2400-2600]:


 66%|██████████████████████████▊              | 131/200 [40:45<05:50,  5.07s/it]

In [ ]:
# Check how many batches are saved
files = session.sql("LIST @LANDSAT_BATCHES").collect()
print(f"{len(files)} batches currently saved:")
for f in files:
    name = f['name'].split('/')[-1]  # Get just the filename
    print(f"   {name}")

Cell 7: Combine batches and also calculating new features,
new features are:
Chlorophyll_Proxy
BSI
SAVI
AWEI_nsh
AWEI_sh
NRI
Turbidity_Index
NDVI
MNDWI
NDMI
NDTI
EVI
NDBI
NBR

In [ ]:
import glob

# Configuration
BATCH_DIR = "landsat_batches"
BATCH_SIZE = 200
total_samples = len(input_df)
total_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

print("Combining batches...\n")

# Load all batch files
batch_files = sorted(glob.glob(f"{BATCH_DIR}/landsat_batch_*.csv"))

if not batch_files:
    print("No batch files found! Run the extraction cell first.")
else:
    print(f"Found {len(batch_files)} batches")
    
    if len(batch_files) != total_batches:
        print(f"Warning: Expected {total_batches}, found {len(batch_files)}")
        missing = total_batches - len(batch_files)
        print(f"Missing {missing} batches - re-run extraction to complete\n")
    
    # Combine all batches
    batch_dfs = [pd.read_csv(f) for f in batch_files]
    landsat_bands = pd.concat(batch_dfs, ignore_index=True)
    
    print(f"✓ Combined shape: {landsat_bands.shape}\n")
    
    # Add location and date columns
    landsat_bands['Latitude'] = input_df['Latitude'].values
    landsat_bands['Longitude'] = input_df['Longitude'].values
    landsat_bands['Sample Date'] = input_df['Sample Date'].values
    
    print("Calculating water quality indices...")
    
    # ... (rest of your index calculations continue here)

Cell 7: Data Quality Checks

In [ ]:
print("DATA QUALITY CHECKS")
print(f"Samples: {len(landsat_df)} / {len(input_df)} expected")
print(f"Complete samples: {landsat_df.dropna().shape[0]} ({100*landsat_df.dropna().shape[0]/len(landsat_df):.1f}%)")
print(f"\nMissing values:\n{landsat_df[bands].isna().sum()}")
print(f"\nBand ranges:\n{landsat_df[bands].describe()}")

Cell 8: Download link

In [ ]:
# Load final dataset
landsat_df = pd.read_csv("/tmp/landsat_features.csv")

# Convert to CSV string
csv_string = landsat_df.to_csv(index=False)

# Encode to base64
b64_encoded = base64.b64encode(csv_string.encode()).decode()

# Create download link
download_link = f'<a href="data:file/csv;base64,{b64_encoded}" download="landsat_features.csv">📥 Download landsat_features.csv</a>'

print(" Landsat Feature Extraction Complete!")
print(f" Final dataset: {len(landsat_df):,} samples × {len(landsat_df.columns)} features")
print(f" File size: {len(csv_string):,} bytes ({len(csv_string)/1024/1024:.2f} MB)")


display(HTML(download_link))
